# Sara Hearts — Tile Stitching & Napari Viewer

Stitches tiled lightsheet MIP acquisitions into a mosaic timeseries for viewing in napari.

**Layout:**
| Stack | Tiles | Sample |
|-------|-------|--------|
| `stack_0` | 4 cols × 5 rows | IK26-46-OFT1and2 |
| `stack_1` | 4 cols × 4 rows | IK26-46-OFT3 |

**Channels:** `Cam_long` (cyan), `Cam_short` (magenta)  
**Timepoints:** 120  
**Files:** `{channel}_{t:05d}.max.z.tiff` in `{tile_folder}/mip/`

**Acquisition snake order** (starting bottom-right, going up, then left):
```
col x03: y04 → y03 → y02 → y01 → y00  (↑)
col x02: y00 → y01 → y02 → y03 → y04  (↓)
col x01: y04 → y03 → y02 → y01 → y00  (↑)
col x00: y00 → y01 → y02 → y03 → y04  (↓)
```

In [1]:
import numpy as np
import tifffile
from pathlib import Path
import re
import napari
import dask.array as da
from dask import delayed
from scipy.ndimage import gaussian_filter

## 1. Configuration

In [2]:
BASE_DIR  = Path(r"Y:\sshah18\IK26_46_OFTs\2026-04-13_181335\raw")
CHANNELS  = ["Cam_long", "Cam_short"]

# ── Per-stack overlap (fraction, 0.0–1.0) ────────────────────────────────────
OVERLAP = {
    'stack_0': {'x': 0.05, 'y': 0.2},
    'stack_1': {'x': 0.13, 'y': 0.3},
}

# ── Per-stack edge crop (pixels, applied after rotation) ─────────────────────
CROP = {
    'stack_0': {'top': 800, 'bottom': 800, 'left': 0, 'right': 0},
    'stack_1': {'top': 800, 'bottom': 800, 'left': 0, 'right': 0},
}

# ── Shading correction ────────────────────────────────────────────────────────
# Estimates and removes vignetting by dividing each tile by a heavily blurred
# version of itself. sigma_fraction controls how broad the blur is (0.1–0.5).
SHADING_CORRECT  = True
SHADING_SIGMA    = 0.2   # fraction of the tile's shorter dimension

# ── Blend width (pixels) ─────────────────────────────────────────────────────
# Overlap regions are feathered with a linear ramp over this many pixels.
# Increase to soften seams; set to 0 to disable (last tile wins).
BLEND_PX = 100

## 2. Discover Tiles

In [3]:
# Parse folder names: stack_0-x02-y03-IK26-46-OFT1and2_channel_0_obj_bottom
_pat = re.compile(r'(stack_\d+)-x(\d+)-y(\d+)')

tiles = {}  # stack_id -> list of (xi, yi, Path)
for folder in sorted(BASE_DIR.iterdir()):
    m = _pat.match(folder.name)
    if m:
        stack = m.group(1)
        xi, yi = int(m.group(2)), int(m.group(3))
        tiles.setdefault(stack, []).append((xi, yi, folder))

for stack_id, tile_list in sorted(tiles.items()):
    xi_vals = sorted({t[0] for t in tile_list})
    yi_vals = sorted({t[1] for t in tile_list})
    n_tp = len(list((tile_list[0][2] / 'mip').glob(f'{CHANNELS[0]}_*.max.z.tiff')))
    print(f"{stack_id}: {len(xi_vals)} cols (x{xi_vals[0]:02d}–x{xi_vals[-1]:02d}) "
          f"× {len(yi_vals)} rows (y{yi_vals[0]:02d}–y{yi_vals[-1]:02d}) "
          f"= {len(tile_list)} tiles  |  {n_tp} timepoints")

print(f"\nChannels: {CHANNELS}")

stack_0: 4 cols (x00–x03) × 5 rows (y00–y04) = 20 tiles  |  60 timepoints
stack_1: 4 cols (x00–x03) × 4 rows (y00–y03) = 16 tiles  |  60 timepoints

Channels: ['Cam_long', 'Cam_short']


## 3. Probe Tile Dimensions

In [4]:
sample_path = tiles['stack_0'][0][2] / 'mip' / f'{CHANNELS[0]}_00000.max.z.tiff'
sample_img  = tifffile.imread(sample_path)

TILE_H, TILE_W = sample_img.shape[-2], sample_img.shape[-1]
TILE_DTYPE     = sample_img.dtype

print(f"Tile size : {TILE_H} × {TILE_W} px")
print(f"dtype     : {TILE_DTYPE}")

Tile size : 2368 × 4432 px
dtype     : uint16


## 4. Stitching Function

Tiles are placed on a regular grid using their `(xi, yi)` folder indices.

- Column `xi` → pixel offset `xi × step_x`  
- Row `yi` → pixel offset `yi × step_y`  
- `step = tile_size × (1 − overlap)`

Where tiles overlap, the **last tile written wins** (use `OVERLAP = 0` for a clean non-overlapping grid).

In [5]:
def crop_tile(tile, crop):
    """Crop pixels from each edge of a 2-D tile."""
    t, b, l, r = crop['top'], crop['bottom'], crop['left'], crop['right']
    h, w = tile.shape[-2], tile.shape[-1]
    return tile[..., t:h - b if b else h, l:w - r if r else w]


def correct_shading(tile_f):
    """Remove vignetting by dividing by a heavily blurred flat-field estimate.

    tile_f : float32 array (H, W)
    Returns float32 array with the same mean intensity.
    """
    sigma = max(10, int(min(tile_f.shape) * SHADING_SIGMA))
    flat  = gaussian_filter(tile_f, sigma=sigma)
    mean  = flat.mean()
    return tile_f * mean / (flat + 1e-3)


def make_weight_map(h, w):
    """Linear ramp from 0 at edges to 1 in centre, clamped to BLEND_PX."""
    if BLEND_PX == 0:
        return np.ones((h, w), dtype=np.float32)
    wy = np.clip(np.minimum(np.arange(h), np.arange(h - 1, -1, -1)), 0, BLEND_PX) / BLEND_PX
    wx = np.clip(np.minimum(np.arange(w), np.arange(w - 1, -1, -1)), 0, BLEND_PX) / BLEND_PX
    return np.outer(wy, wx).astype(np.float32)


def stitch_timepoint(stack_id, channel, timepoint, overlap=None, crop=None):
    """Return a stitched 2-D mosaic for one stack / channel / timepoint.

    Pipeline per tile:
      1. Read → rotate 90° CW → crop edges
      2. Shading correction (optional)
      3. Weighted accumulation for linear feather blending
    """
    if overlap is None:
        overlap = OVERLAP[stack_id]
    if crop is None:
        crop = CROP[stack_id]

    tile_list = tiles[stack_id]
    xi_max = max(t[0] for t in tile_list)
    yi_max = max(t[1] for t in tile_list)

    tp_str  = f'{timepoint:05d}'
    _sample = crop_tile(np.rot90(tifffile.imread(
        tile_list[0][2] / 'mip' / f'{channel}_{tp_str}.max.z.tiff'), k=-1), crop)
    rot_h, rot_w = _sample.shape[-2], _sample.shape[-1]
    orig_dtype   = _sample.dtype

    step_x = int(rot_w * (1 - overlap['x']))
    step_y = int(rot_h * (1 - overlap['y']))
    out_h  = yi_max * step_y + rot_h
    out_w  = xi_max * step_x + rot_w

    mosaic_sum = np.zeros((out_h, out_w), dtype=np.float32)
    mosaic_wgt = np.zeros((out_h, out_w), dtype=np.float32)
    wmap = make_weight_map(rot_h, rot_w)

    for xi, yi, folder in tile_list:
        fpath  = folder / 'mip' / f'{channel}_{tp_str}.max.z.tiff'
        tile   = crop_tile(np.rot90(tifffile.imread(fpath), k=-1), crop)
        tile_f = tile.astype(np.float32)
        if SHADING_CORRECT:
            tile_f = correct_shading(tile_f)
        col = (xi_max - xi) * step_x
        row = yi * step_y
        mosaic_sum[row:row + rot_h, col:col + rot_w] += tile_f * wmap
        mosaic_wgt[row:row + rot_h, col:col + rot_w] += wmap

    valid  = mosaic_wgt > 0
    result = np.where(valid, mosaic_sum / np.where(valid, mosaic_wgt, 1), 0)

    if np.issubdtype(orig_dtype, np.integer):
        info   = np.iinfo(orig_dtype)
        result = result.clip(info.min, info.max)

    return result.astype(orig_dtype)

## 5. Quick Preview — Single Timepoint in Napari

Use this cell to tune `OVERLAP` before loading the full timeseries.

In [6]:
# ── Parameters ────────────────────────────────────────────────────────────────
PREVIEW_STACK = 'stack_1'
PREVIEW_T     = 1
# ─────────────────────────────────────────────────────────────────────────────

long  = stitch_timepoint(PREVIEW_STACK, 'Cam_long',  PREVIEW_T)
short = stitch_timepoint(PREVIEW_STACK, 'Cam_short', PREVIEW_T)

v = napari.Viewer()
v.add_image(long,  name='Cam_long',  colormap='magenta',    blending='additive')
v.add_image(short, name='Cam_short', colormap='cyan', blending='additive')

print(f"Mosaic size: {long.shape[0]} × {long.shape[1]} px")

Mosaic size: 8778 × 8548 px


## 6. Build Dask Arrays (Lazy — All Timepoints)

Creates a `(T, C, Y, X)` dask array per stack.  
Tiles are loaded on demand as napari scrolls through time.

In [7]:
def build_dask_stack(stack_id):
    """Build a lazy (T, C, Y, X) dask array for the given stack."""
    overlap   = OVERLAP[stack_id]
    crop      = CROP[stack_id]
    tile_list = tiles[stack_id]
    xi_max    = max(t[0] for t in tile_list)
    yi_max    = max(t[1] for t in tile_list)

    # Use cropped + rotated shape to match stitch_timepoint exactly
    _sample    = crop_tile(np.rot90(tifffile.imread(
        tile_list[0][2] / 'mip' / f'{CHANNELS[0]}_00000.max.z.tiff'), k=-1), crop)
    rot_h, rot_w = _sample.shape[-2], _sample.shape[-1]
    tile_dtype   = _sample.dtype

    step_x = int(rot_w * (1 - overlap['x']))
    step_y = int(rot_h * (1 - overlap['y']))
    out_h  = yi_max * step_y + rot_h
    out_w  = xi_max * step_x + rot_w

    n_tp = len(list((tile_list[0][2] / 'mip').glob(f'{CHANNELS[0]}_*.max.z.tiff')))

    lazy_frames = []
    for t in range(n_tp):
        lazy_channels = []
        for ch in CHANNELS:
            arr = da.from_delayed(
                delayed(stitch_timepoint)(stack_id, ch, t),
                shape=(out_h, out_w),
                dtype=tile_dtype,
            )
            lazy_channels.append(arr)
        lazy_frames.append(da.stack(lazy_channels))  # (C, Y, X)

    return da.stack(lazy_frames)  # (T, C, Y, X)


dask_stack0 = build_dask_stack('stack_0')
dask_stack1 = build_dask_stack('stack_1')

print(f"stack_0 shape: {dask_stack0.shape}  (T, C, Y, X)")
print(f"stack_1 shape: {dask_stack1.shape}  (T, C, Y, X)")

stack_0 shape: (60, 2, 11892, 9115)  (T, C, Y, X)
stack_1 shape: (60, 2, 8778, 8548)  (T, C, Y, X)


## 7. Napari Viewer

In [8]:
viewer = napari.Viewer()

viewer.add_image(
    dask_stack0,
    name='stack_0 — OFT1&2',
    channel_axis=1,
    colormap=['magenta', 'cyan'],
    blending='additive',
)

viewer.add_image(
    dask_stack1,
    name='stack_1 — OFT3',
    channel_axis=1,
    colormap=['magenta', 'cyan'],
    blending='additive',
)

napari.run()

---
## 8. Export

Writes each stack to the `Processed/` folder inside `BASE_DIR`:

| File | Description |
|------|-------------|
| `{stack}_{channel}_full.tiff` | Full-resolution 16-bit BigTIFF, all timepoints |
| `{stack}_{channel}_ds{N}_8bit.tiff` | Downsampled 8-bit TIFF, all timepoints |
| `{stack}_composite_ds{N}.avi` | Downsampled composite colour AVI with timestamps |

Timestamps start at `00:00` and increment by `TIME_INTERVAL_MIN` minutes per frame.

In [9]:
# ── Export parameters ─────────────────────────────────────────────────────────
DOWNSAMPLE        = 4     # spatial factor for 8-bit TIFF and AVI (e.g. 4 → ¼ size)
TIME_INTERVAL_MIN = 12    # minutes between timepoints
VIDEO_FPS         = 2     # playback speed for AVI
EXPORT_STACKS     = ['stack_0', 'stack_1']

# ── Output folder ─────────────────────────────────────────────────────────────
# Set to None to auto-create a "Processed" folder inside BASE_DIR, or provide
# a custom path. NOTE: do not end the path with a backslash.
PROCESSED_DIR = r"D:\Sara\Processed-Ik26_46OFTs-2026-04-13"

# ── Resolve and create ────────────────────────────────────────────────────────
if PROCESSED_DIR is None:
    PROCESSED_DIR = BASE_DIR / "Processed"
else:
    PROCESSED_DIR = Path(PROCESSED_DIR)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output folder: {PROCESSED_DIR}")

Output folder: D:\Sara\Processed-Ik26_46OFTs-2026-04-13


In [10]:
import sys
!{sys.executable} -m pip install imageio-ffmpeg -q
print("imageio-ffmpeg ready")

imageio-ffmpeg ready


In [11]:
import imageio
from PIL import Image, ImageDraw


# ── Helpers ───────────────────────────────────────────────────────────────────

def make_timestamp(t):
    """Return 'HH:MM' string for timepoint t at TIME_INTERVAL_MIN intervals."""
    total = t * TIME_INTERVAL_MIN
    return f"{total // 60:02d}:{total % 60:02d}"


def downsample_img(img, factor):
    """Block-average downsample a 2-D array by an integer factor."""
    h, w    = img.shape
    h_new   = h // factor
    w_new   = w // factor
    trimmed = img[:h_new * factor, :w_new * factor].astype(np.float32)
    return trimmed.reshape(h_new, factor, w_new, factor).mean((1, 3))


def to_uint8(img, vmin=None, vmax=None):
    """Normalise to [0, 255] uint8. vmin/vmax default to image min/max."""
    f    = img.astype(np.float32)
    vmin = f.min() if vmin is None else vmin
    vmax = f.max() if vmax is None else vmax
    denom = vmax - vmin if vmax > vmin else 1.0
    return np.clip((f - vmin) / denom * 255, 0, 255).astype(np.uint8)


def stamp_frame(rgb_u8, text, pos=(14, 14), font_size=32):
    """Burn a white timestamp with black shadow into an RGB uint8 frame."""
    img  = Image.fromarray(rgb_u8)
    draw = ImageDraw.Draw(img)
    try:
        from PIL import ImageFont
        font = ImageFont.truetype("arial.ttf", font_size)
    except Exception:
        font = None
    ox, oy = pos
    for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        draw.text((ox + dx, oy + dy), text, fill=(0, 0, 0), font=font)
    draw.text(pos, text, fill=(255, 255, 255), font=font)
    return np.array(img)


def make_composite_rgb(long_u8, short_u8):
    """Cam_long → green channel, Cam_short → magenta (R + B)."""
    return np.stack([short_u8, long_u8, short_u8], axis=-1)


# ── Per-stack normalisation bounds ────────────────────────────────────────────

def get_norm_bounds(stack_id):
    """Return {channel: (vmin, vmax)} using 0.1 / 99.9 percentile clipping."""
    bounds = {}
    for ch in CHANNELS:
        frame   = stitch_timepoint(stack_id, ch, 0)
        nonzero = frame[frame > 0]
        vmin    = float(np.percentile(nonzero,  0.1))
        vmax    = float(np.percentile(nonzero, 99.9))
        bounds[ch] = (vmin, vmax)
        print(f"  {stack_id} {ch}: vmin={vmin:.0f}  vmax={vmax:.0f}")
    return bounds


# ── Main export routine ───────────────────────────────────────────────────────

def export_stack(stack_id):
    tile_list = tiles[stack_id]
    n_tp = len(list(
        (tile_list[0][2] / 'mip').glob(f'{CHANNELS[0]}_*.max.z.tiff')))

    print(f"\n{'='*60}")
    print(f"Exporting {stack_id}  ({n_tp} timepoints)")
    print(f"{'='*60}")

    print("Computing normalisation bounds (0.1 / 99.9 percentile) …")
    bounds = get_norm_bounds(stack_id)

    # --- open writers ---
    tiff_writers  = {}
    tiff8_writers = {}
    for ch in CHANNELS:
        tiff_writers[ch] = tifffile.TiffWriter(
            PROCESSED_DIR / f"{stack_id}_{ch}_full.tiff", bigtiff=True)
        tiff8_writers[ch] = tifffile.TiffWriter(
            PROCESSED_DIR / f"{stack_id}_{ch}_ds{DOWNSAMPLE}_8bit.tiff",
            bigtiff=True)

    avi_path   = PROCESSED_DIR / f"{stack_id}_composite_ds{DOWNSAMPLE}.avi"
    avi_writer = imageio.get_writer(
        str(avi_path), format='ffmpeg', fps=VIDEO_FPS,
        codec='mjpeg', quality=9, pixelformat='yuvj444p')

    # --- loop over timepoints ---
    for t in range(n_tp):
        ts = make_timestamp(t)
        print(f"  t={t:3d}  [{ts}]", end='\r')

        frames = {}
        for ch in CHANNELS:
            mosaic = stitch_timepoint(stack_id, ch, t)

            # full-res 16-bit
            tiff_writers[ch].write(
                mosaic, description=f"t={t} time={ts}", contiguous=True)

            # downsampled 8-bit
            ds  = downsample_img(mosaic, DOWNSAMPLE)
            u8  = to_uint8(ds, *bounds[ch])
            tiff8_writers[ch].write(u8, contiguous=True)
            frames[ch] = u8

        # composite AVI frame with timestamp
        rgb = make_composite_rgb(frames['Cam_long'], frames['Cam_short'])
        rgb = stamp_frame(rgb, ts)
        avi_writer.append_data(rgb)

    # --- close all writers ---
    for w in tiff_writers.values():
        w.close()
    for w in tiff8_writers.values():
        w.close()
    avi_writer.close()

    print(f"\n  ✓ Saved to: {PROCESSED_DIR}")

In [12]:
# Run export for all stacks listed in EXPORT_STACKS
for stack_id in EXPORT_STACKS:
    export_stack(stack_id)

print("\nAll done.")


Exporting stack_0  (60 timepoints)
Computing normalisation bounds (0.1 / 99.9 percentile) …
  stack_0 Cam_long: vmin=190  vmax=1191
  stack_0 Cam_short: vmin=204  vmax=456
  t=  0  [00:00]

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (2278, 2973) to (2288, 2976) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


  t= 59  [11:48]
  ✓ Saved to: D:\Sara\Processed-Ik26_46OFTs-2026-04-13

Exporting stack_1  (60 timepoints)
Computing normalisation bounds (0.1 / 99.9 percentile) …
  stack_1 Cam_long: vmin=181  vmax=1459
  stack_1 Cam_short: vmin=210  vmax=341
  t=  0  [00:00]

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (2137, 2194) to (2144, 2208) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


  t= 59  [11:48]
  ✓ Saved to: D:\Sara\Processed-Ik26_46OFTs-2026-04-13

All done.
